# Objectives

### English

- Reuse the refactored inference pipeline available in the src package.

- Implement reusable Monte Carlo simulation utilities for World Cup matches and knockout rounds.

- Simulate individual matches using model probability distributions instead of deterministic predictions.

- Build a complete tournament simulation pipeline for the remaining 2026 World Cup matches.

- Estimate team advancement and championship probabilities through repeated tournament simulations.

- Compare simulation outcomes produced by Random Forest, XGBoost, and an ensemble approach.

- Generate structured probability tables for tournament progression and final championship predictions.

- Validate that the simulation layer integrates correctly with the reusable feature engineering and artifact-loading architecture.

- Complete Phase I of the World Cup Predictor 2026 project with a reusable tournament simulation module.


### Español

- Reutilizar el pipeline de inferencia refactorizado disponible dentro del paquete src.

- Implementar utilidades reutilizables de simulación Monte Carlo para partidos y rondas eliminatorias del Mundial.

- Simular partidos individuales utilizando distribuciones de probabilidad de los modelos en lugar de predicciones determinísticas.

- Construir un pipeline completo de simulación para los partidos restantes del Mundial 2026.

- Estimar las probabilidades de avance y campeonato de cada selección mediante simulaciones repetidas del torneo.

- Comparar los resultados generados por Random Forest, XGBoost y un enfoque ensemble.

- Generar tablas estructuradas de probabilidades de progresión en el torneo y predicciones finales de campeón.

- Validar la correcta integración de la capa de simulación con la arquitectura reutilizable de carga de artefactos e ingeniería de variables.

- Completar la Fase I del proyecto World Cup Predictor 2026 mediante un módulo reutilizable de simulación del torneo.


# Configuration

## Imports

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
import numpy as np
import pandas as pd

from src.artifacts import (
    load_model,
    load_dataframe
)

from src.feature_engineering import (
    build_prediction_ready_match_vector)

## Paths

In [4]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

PROCESSED_DATA_DIR = DATA_DIR / "processed"

## Load Models

In [5]:
rf_model = load_model(
    MODELS_DIR / "random_forest_v03.pkl"
)

xgb_model = load_model(
    MODELS_DIR / "xgboost_v04.pkl"
)

print("Models loaded successfully.")

Models loaded successfully.


## Load Current Team Vectors

In [6]:
current_team_vectors = load_dataframe(
    PROCESSED_DATA_DIR / "current_team_vectors_v01.csv"
)

print(f"Current team vectors: {current_team_vectors.shape}")

Current team vectors: (48, 32)


## Import Simulation Modules


In [7]:
# Simulation modules will be imported after the reusable
# functions are implemented and exported to src/simulation.py.

# Monte Carlo Utilities

## Simulate Single Match

This section implements the fundamental unit of the Monte Carlo pipeline.

Instead of selecting the class with the highest predicted probability, the
simulation randomly samples one possible match outcome according to the model's
probability distribution.

This preserves model uncertainty and allows repeated simulations to produce
different tournament outcomes.

In [50]:
def simulate_match(
    home_team,
    away_team,
    model,
    current_team_vectors,
    statistical_columns,
    feature_names,
    target_classes=None,
    rng=None
):
    """
    Simulate one match by sampling from the model's predicted probability
    distribution.

    Parameters
    ----------
    home_team : str
        Name of the home team.

    away_team : str
        Name of the away team.

    model : estimator
        Trained classification model implementing predict_proba().

    current_team_vectors : pandas.DataFrame
        Current team-level vectors used to construct match features.

    statistical_columns : list-like
        Statistical columns used during feature engineering.

    feature_names : list-like
        Ordered feature schema expected by the model.

    target_classes : list-like, optional
        Original target classes represented by the model's internal classes.
        This is especially necessary for models such as XGBoost that may use
        encoded classes such as [0, 1, 2].

    rng : numpy.random.Generator, optional
        Random number generator used for reproducible simulation.

    Returns
    -------
    dict
        Simulated match result, winner, predicted probabilities, and the
        model-ready prediction vector.
    """

    if rng is None:
        rng = np.random.default_rng()

    prediction_vector = build_prediction_ready_match_vector(
        home_team=home_team,
        away_team=away_team,
        team_vectors=current_team_vectors,
        statistical_columns=statistical_columns,
        historical_feature_columns=feature_names,
        team_id_column="country_clean"
    )

    model_prediction_vector = prediction_vector.loc[
        :,
        list(feature_names)
    ].copy()

    assert (
        model_prediction_vector.columns.tolist()
        == list(feature_names)
    ), (
        "Prediction vector feature schema does not match "
        "the model schema."
    )

    probabilities = model.predict_proba(
        model_prediction_vector
    )[0]

    model_classes = list(model.classes_)

    if target_classes is None:
        decoded_classes = model_classes

    else:
        target_classes = list(target_classes)

        if len(model_classes) != len(target_classes):
            raise ValueError(
                "The number of model classes does not match "
                "the number of target classes."
            )

        decoded_classes = target_classes

    probability_by_target = {
        int(target_class): float(probability)
        for target_class, probability in zip(
            decoded_classes,
            probabilities
        )
    }

    sampled_target = int(
        rng.choice(
            decoded_classes,
            p=probabilities
        )
    )

    result_mapping = {
        1: "Home Win",
        0: "Draw",
        -1: "Away Win"
    }

    if sampled_target not in result_mapping:
        raise ValueError(
            f"Unsupported decoded target class: {sampled_target}"
        )

    simulated_result = result_mapping[
        sampled_target
    ]

    if sampled_target == 1:
        winner = home_team

    elif sampled_target == -1:
        winner = away_team

    else:
        winner = None

    probability_mapping = {
        "Home Win": probability_by_target.get(1, 0.0),
        "Draw": probability_by_target.get(0, 0.0),
        "Away Win": probability_by_target.get(-1, 0.0)
    }

    return {
        "home_team": str(home_team),
        "away_team": str(away_team),
        "simulated_class": sampled_target,
        "simulated_result": simulated_result,
        "winner": (
            None
            if winner is None
            else str(winner)
        ),
        "probabilities": probability_mapping,
        "prediction_vector": model_prediction_vector
    }

In [9]:
rf_artifact = rf_model
xgb_artifact = xgb_model

rf_model = rf_artifact["model"]
xgb_model = xgb_artifact["model"]

rf_features = rf_artifact["feature_names"]
xgb_features = xgb_artifact["feature_names"]

In [10]:
print("RF model:", type(rf_model).__name__)
print("XGB model:", type(xgb_model).__name__)

print("RF feature count:", len(rf_features))
print("XGB feature count:", len(xgb_features))

RF model: RandomForestClassifier
XGB model: XGBClassifier
RF feature count: 124
XGB feature count: 124


### Define Statistical Columns

In [11]:
TEAM_METADATA_COLUMNS = [
    "country_clean",
    "squad_size",
    "matched_players",
    "unmatched_players",
    "coverage"
]

statistical_columns = [
    column
    for column in current_team_vectors.columns
    if column not in TEAM_METADATA_COLUMNS
]

print("Statistical columns:", len(statistical_columns))

Statistical columns: 27


### Test Single Match Simulation

In [12]:
test_rng = np.random.default_rng(42)

rf_test_simulation = simulate_match(
    home_team="Argentina",
    away_team="Brazil",
    model=rf_model,
    current_team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
    feature_names=rf_features,
    rng=test_rng
)

print(
    "Match:",
    rf_test_simulation["home_team"],
    "vs",
    rf_test_simulation["away_team"]
)

print("Simulated result:", rf_test_simulation["simulated_result"])
print("Winner:", rf_test_simulation["winner"])
print("Probabilities:", rf_test_simulation["probabilities"])

Match: Argentina vs Brazil
Simulated result: Home Win
Winner: Argentina
Probabilities: {'Away Win': 0.36, 'Draw': 0.355, 'Home Win': 0.285}


In [13]:
assert rf_test_simulation["prediction_vector"].shape[0] == 1

assert (
    rf_test_simulation["prediction_vector"].columns.tolist()
    == list(rf_features)
)

assert np.isclose(
    sum(rf_test_simulation["probabilities"].values()),
    1.0
)

assert rf_test_simulation["simulated_result"] in {
    "Home Win",
    "Draw",
    "Away Win"
}

assert rf_test_simulation["winner"] in {
    "Argentina",
    "Brazil",
    None
}

print("Single match simulation test passed.")

Single match simulation test passed.


## Simulate Knockout Match

Knockout matches require a winner, even when the simulated regulation-time
result is a draw.

This function first simulates the match through `simulate_match()`. If the
sampled result is a draw, the winner is selected by normalizing the model's
home-win and away-win probabilities while excluding the draw probability.

This provides a simple probabilistic approximation for extra time and penalty
shootouts while preserving the relative strength estimated by the model.

In [57]:
def simulate_knockout_match(
    home_team,
    away_team,
    model,
    current_team_vectors,
    statistical_columns,
    feature_names,
    target_classes=None,
    rng=None
):
    """
    Simulate a knockout match and guarantee that one team advances.

    The regulation-time result is generated through simulate_match(). If the
    result is a draw, the function resolves the tie by normalizing the model's
    home-win and away-win probabilities and sampling one advancing team.

    Parameters
    ----------
    home_team : str
        Name of the home team.

    away_team : str
        Name of the away team.

    model : estimator
        Trained classification model implementing predict_proba().

    current_team_vectors : pandas.DataFrame
        Current team-level vectors used to construct match features.

    statistical_columns : list-like
        Statistical columns used during feature engineering.

    feature_names : list-like
        Ordered feature schema expected by the model.

    rng : numpy.random.Generator, optional
        Random number generator used for reproducible simulations.

    Returns
    -------
    dict
        Complete knockout match result, including the regulation-time outcome,
        winner, loser, tie-break status, and tie-break probabilities.
    """

    if rng is None:
        rng = np.random.default_rng()

    match_result = simulate_match(
        home_team=home_team,
        away_team=away_team,
        model=model,
        current_team_vectors=current_team_vectors,
        statistical_columns=statistical_columns,
        feature_names=feature_names,
        target_classes=target_classes,
        rng=rng
    )

    regulation_winner = match_result["winner"]
    decided_by_tiebreak = regulation_winner is None

    if not decided_by_tiebreak:
        winner = regulation_winner
        tiebreak_probabilities = None

    else:
        home_win_probability = (
            match_result["probabilities"]["Home Win"]
        )

        away_win_probability = (
            match_result["probabilities"]["Away Win"]
        )

        total_win_probability = (
            home_win_probability
            + away_win_probability
        )

        if np.isclose(total_win_probability, 0.0):
            home_advance_probability = 0.5
            away_advance_probability = 0.5

        else:
            home_advance_probability = (
                home_win_probability
                / total_win_probability
            )

            away_advance_probability = (
                away_win_probability
                / total_win_probability
            )

        winner = rng.choice(
            [home_team, away_team],
            p=[
                home_advance_probability,
                away_advance_probability
            ]
        )

        tiebreak_probabilities = {
            home_team: float(home_advance_probability),
            away_team: float(away_advance_probability)
        }

    winner = str(winner)

    loser = str(
        away_team
        if winner == home_team
        else home_team
    )

    return {
        **match_result,
        "winner": winner,
        "loser": loser,
        "decided_by_tiebreak": bool(decided_by_tiebreak),
        "tiebreak_probabilities": tiebreak_probabilities
    }

### Test Knockout Match Simulation

In [22]:
test_rng = np.random.default_rng(42)

rf_knockout_test = simulate_knockout_match(
    home_team="Argentina",
    away_team="Brazil",
    model=rf_model,
    current_team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
    feature_names=rf_features,
    rng=test_rng
)

print(
    "Match:",
    rf_knockout_test["home_team"],
    "vs",
    rf_knockout_test["away_team"]
)

print(
    "Regulation result:",
    rf_knockout_test["simulated_result"]
)

print(
    "Decided by tiebreak:",
    rf_knockout_test["decided_by_tiebreak"]
)

print(
    "Winner:",
    rf_knockout_test["winner"]
)

print(
    "Loser:",
    rf_knockout_test["loser"]
)

print(
    "Tiebreak probabilities:",
    rf_knockout_test["tiebreak_probabilities"]
)

Match: Argentina vs Brazil
Regulation result: Home Win
Decided by tiebreak: False
Winner: Argentina
Loser: Brazil
Tiebreak probabilities: None


In [23]:
assert rf_knockout_test["winner"] in {
    "Argentina",
    "Brazil"
}

assert rf_knockout_test["loser"] in {
    "Argentina",
    "Brazil"
}

assert (
    rf_knockout_test["winner"]
    != rf_knockout_test["loser"]
)

assert isinstance(
    rf_knockout_test["winner"],
    str
)

assert isinstance(
    rf_knockout_test["loser"],
    str
)

assert isinstance(
    rf_knockout_test["decided_by_tiebreak"],
    bool
)

if rf_knockout_test["decided_by_tiebreak"]:

    assert (
        rf_knockout_test["tiebreak_probabilities"]
        is not None
    )

    assert np.isclose(
        sum(
            rf_knockout_test[
                "tiebreak_probabilities"
            ].values()
        ),
        1.0
    )

else:

    assert (
        rf_knockout_test["tiebreak_probabilities"]
        is None
    )

print("Knockout match simulation test passed.")

Knockout match simulation test passed.


## Simulate Round

In [60]:
def simulate_round(
    fixtures,
    model,
    current_team_vectors,
    statistical_columns,
    feature_names,
    target_classes=None,
    rng=None
):
    """
    Simulate one complete knockout round.

    Parameters
    ----------
    fixtures : list of tuple
        Sequence of knockout pairings in the form:
        [(home_team, away_team), ...].

    model : estimator
        Trained classification model implementing predict_proba().

    current_team_vectors : pandas.DataFrame
        Current team-level vectors used to construct match features.

    statistical_columns : list-like
        Statistical columns used during feature engineering.

    feature_names : list-like
        Ordered feature schema expected by the model.

    rng : numpy.random.Generator, optional
        Random number generator used for reproducible simulations.

    Returns
    -------
    dict
        Round result containing the original fixtures, complete match results,
        and the ordered list of qualified teams.
    """

    if rng is None:
        rng = np.random.default_rng()

    if not fixtures:
        raise ValueError(
            "fixtures must contain at least one knockout match."
        )

    match_results = []
    qualified_teams = []

    for fixture in fixtures:

        if len(fixture) != 2:
            raise ValueError(
                "Each fixture must contain exactly two teams."
            )

        home_team, away_team = fixture

        if home_team == away_team:
            raise ValueError(
                "A team cannot play against itself."
            )

        match_result = simulate_knockout_match(
            home_team=home_team,
            away_team=away_team,
            model=model,
            current_team_vectors=current_team_vectors,
            statistical_columns=statistical_columns,
            feature_names=feature_names,
            target_classes=target_classes,
            rng=rng
        )

        match_results.append(match_result)
        qualified_teams.append(match_result["winner"])

    return {
        "fixtures": list(fixtures),
        "match_results": match_results,
        "qualified_teams": qualified_teams
    }

### Test Round Simulation

In [24]:
test_fixtures = [
    ("Argentina", "Brazil"),
    ("France", "England")
]

test_rng = np.random.default_rng(42)

rf_round_test = simulate_round(
    fixtures=test_fixtures,
    model=rf_model,
    current_team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
    feature_names=rf_features,
    rng=test_rng
)

print("Fixtures:")
for fixture in rf_round_test["fixtures"]:
    print(f"  {fixture[0]} vs {fixture[1]}")

print("\nResults:")
for result in rf_round_test["match_results"]:
    print(
        f"  {result['home_team']} vs {result['away_team']} "
        f"→ {result['winner']}"
    )

print(
    "\nQualified teams:",
    rf_round_test["qualified_teams"]
)

Fixtures:
  Argentina vs Brazil
  France vs England

Results:
  Argentina vs Brazil → Argentina
  France vs England → England

Qualified teams: ['Argentina', 'England']


In [25]:
assert len(
    rf_round_test["fixtures"]
) == len(test_fixtures)

assert len(
    rf_round_test["match_results"]
) == len(test_fixtures)

assert len(
    rf_round_test["qualified_teams"]
) == len(test_fixtures)

assert all(
    result["winner"] is not None
    for result in rf_round_test["match_results"]
)

assert all(
    qualified_team in fixture
    for qualified_team, fixture in zip(
        rf_round_test["qualified_teams"],
        test_fixtures
    )
)

assert (
    rf_round_test["qualified_teams"]
    == [
        result["winner"]
        for result in rf_round_test["match_results"]
    ]
)

print("Knockout round simulation test passed.")

Knockout round simulation test passed.


## Simulate Tournament

In [61]:
def simulate_tournament(
    initial_fixtures,
    model,
    current_team_vectors,
    statistical_columns,
    feature_names,
    target_classes=None,
    rng=None
):
    """
    Simulate a complete single-elimination tournament.

    Parameters
    ----------
    initial_fixtures : list of tuple
        Initial knockout pairings in the form:
        [(home_team, away_team), ...].

    model : estimator
        Trained classification model implementing predict_proba().

    current_team_vectors : pandas.DataFrame
        Current team-level vectors used to construct match features.

    statistical_columns : list-like
        Statistical columns used during feature engineering.

    feature_names : list-like
        Ordered feature schema expected by the model.

    rng : numpy.random.Generator, optional
        Random number generator used for reproducible simulations.

    Returns
    -------
    dict
        Complete tournament result containing all rounds, the finalists,
        runner-up, and champion.
    """

    if rng is None:
        rng = np.random.default_rng()

    if not initial_fixtures:
        raise ValueError(
            "initial_fixtures must contain at least one match."
        )

    initial_teams = [
        team
        for fixture in initial_fixtures
        for team in fixture
    ]

    number_of_teams = len(initial_teams)

    if number_of_teams < 2:
        raise ValueError(
            "A tournament must contain at least two teams."
        )

    if number_of_teams & (number_of_teams - 1) != 0:
        raise ValueError(
            "The number of teams must be a power of two."
        )

    if len(set(initial_teams)) != number_of_teams:
        raise ValueError(
            "Each team must appear exactly once in the initial fixtures."
        )

    tournament_rounds = []
    current_fixtures = list(initial_fixtures)
    round_number = 1

    while current_fixtures:

        round_result = simulate_round(
            fixtures=current_fixtures,
            model=model,
            current_team_vectors=current_team_vectors,
            statistical_columns=statistical_columns,
            feature_names=feature_names,
            target_classes=target_classes,
            rng=rng
        )

        round_result["round_number"] = round_number
        tournament_rounds.append(round_result)

        qualified_teams = round_result["qualified_teams"]

        if len(qualified_teams) == 1:
            break

        current_fixtures = [
            (
                qualified_teams[index],
                qualified_teams[index + 1]
            )
            for index in range(
                0,
                len(qualified_teams),
                2
            )
        ]

        round_number += 1

    final_result = tournament_rounds[-1]["match_results"][0]

    champion = final_result["winner"]
    runner_up = final_result["loser"]

    return {
        "initial_teams": initial_teams,
        "number_of_teams": number_of_teams,
        "number_of_rounds": len(tournament_rounds),
        "rounds": tournament_rounds,
        "finalists": [
            final_result["home_team"],
            final_result["away_team"]
        ],
        "champion": champion,
        "runner_up": runner_up
    }

### Test Tournament Simulation

In [ ]:
test_tournament_fixtures = [
    ("Argentina", "Brazil"),
    ("France", "England"),
    ("Spain", "Portugal"),
    ("Germany", "Netherlands")
]

test_rng = np.random.default_rng(42)

rf_tournament_test = simulate_tournament(
    initial_fixtures=test_tournament_fixtures,
    model=rf_model,
    current_team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
    feature_names=rf_features,
    rng=test_rng
)

print(
    "Number of teams:",
    rf_tournament_test["number_of_teams"]
)

print(
    "Number of rounds:",
    rf_tournament_test["number_of_rounds"]
)

for round_result in rf_tournament_test["rounds"]:

    print(
        f"\nRound {round_result['round_number']}"
    )

    for match_result in round_result["match_results"]:

        print(
            f"  {match_result['home_team']} "
            f"vs {match_result['away_team']} "
            f"→ {match_result['winner']}"
        )

print(
    "\nFinalists:",
    rf_tournament_test["finalists"]
)

print(
    "Champion:",
    rf_tournament_test["champion"]
)

print(
    "Runner-up:",
    rf_tournament_test["runner_up"]
)

Number of teams: 8
Number of rounds: 3

Round 1
  Argentina vs Brazil → Argentina
  France vs England → England
  Spain vs Portugal → Spain
  Germany vs Netherlands → Netherlands

Round 2
  Argentina vs England → Argentina
  Spain vs Netherlands → Spain

Round 3
  Argentina vs Spain → Argentina

Finalists: ['Argentina', 'Spain']
Champion: Argentina
Runner-up: Spain


In [ ]:
assert (
    rf_tournament_test["number_of_teams"]
    == 8
)

assert (
    rf_tournament_test["number_of_rounds"]
    == 3
)

assert len(
    rf_tournament_test["rounds"]
) == 3

assert len(
    rf_tournament_test["rounds"][0]["match_results"]
) == 4

assert len(
    rf_tournament_test["rounds"][1]["match_results"]
) == 2

assert len(
    rf_tournament_test["rounds"][2]["match_results"]
) == 1

assert len(
    rf_tournament_test["finalists"]
) == 2

assert (
    rf_tournament_test["champion"]
    in rf_tournament_test["finalists"]
)

assert (
    rf_tournament_test["runner_up"]
    in rf_tournament_test["finalists"]
)

assert (
    rf_tournament_test["champion"]
    != rf_tournament_test["runner_up"]
)

assert isinstance(
    rf_tournament_test["champion"],
    str
)

assert isinstance(
    rf_tournament_test["runner_up"],
    str
)

assert (
    rf_tournament_test["rounds"][-1][
        "qualified_teams"
    ]
    == [rf_tournament_test["champion"]]
)

print("Tournament simulation test passed.")

Tournament simulation test passed.


## Run Monte Carlo

A single tournament simulation represents only one possible realization of the
model's predicted probability distributions.

Monte Carlo simulation repeats the complete knockout tournament many times and
stores every result independently.

The function is responsible only for generating simulations. Probability
estimation and tournament analytics are performed separately.

In [58]:
def run_monte_carlo(
    initial_fixtures,
    model,
    current_team_vectors,
    statistical_columns,
    feature_names,
    target_classes=None,
    n_simulations=1000,
    random_state=None
):
    """
    Run repeated Monte Carlo simulations of a knockout tournament.

    Parameters
    ----------
    initial_fixtures : list of tuple
        Initial knockout pairings in the form:
        [(home_team, away_team), ...].

    model : estimator
        Trained classification model implementing predict_proba().

    current_team_vectors : pandas.DataFrame
        Current team-level vectors used to construct match features.

    statistical_columns : list-like
        Statistical columns used during feature engineering.

    feature_names : list-like
        Ordered feature schema expected by the model.

    n_simulations : int, default=1000
        Number of complete tournaments to simulate.

    random_state : int, optional
        Seed used to initialize the NumPy random number generator.

    Returns
    -------
    dict
        Monte Carlo metadata and the complete result of every simulated
        tournament.
    """

    if not isinstance(n_simulations, int):
        raise TypeError(
            "n_simulations must be an integer."
        )

    if n_simulations <= 0:
        raise ValueError(
            "n_simulations must be greater than zero."
        )

    if not initial_fixtures:
        raise ValueError(
            "initial_fixtures must contain at least one match."
        )

    rng = np.random.default_rng(random_state)

    initial_teams = [
        str(team)
        for fixture in initial_fixtures
        for team in fixture
    ]

    tournament_results = []

    for simulation_number in range(
        1,
        n_simulations + 1
    ):

        tournament_result = simulate_tournament(
            initial_fixtures=initial_fixtures,
            model=model,
            current_team_vectors=current_team_vectors,
            statistical_columns=statistical_columns,
            feature_names=feature_names,
            target_classes=target_classes,
            rng=rng
        )

        tournament_results.append(
            {
                "simulation_number": simulation_number,
                **tournament_result
            }
        )

    return {
        "n_simulations": n_simulations,
        "random_state": random_state,
        "initial_teams": initial_teams,
        "tournament_results": tournament_results
    }

### Test Monte Carlo Simulation

In [35]:
rf_monte_carlo_test = run_monte_carlo(
    initial_fixtures=test_tournament_fixtures,
    model=rf_model,
    current_team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
    feature_names=rf_features,
    n_simulations=100,
    random_state=42
)

print(
    "Number of simulations:",
    rf_monte_carlo_test["n_simulations"]
)

print(
    "Stored tournaments:",
    len(
        rf_monte_carlo_test[
            "tournament_results"
        ]
    )
)

print(
    "First champion:",
    rf_monte_carlo_test[
        "tournament_results"
    ][0]["champion"]
)

print(
    "First runner-up:",
    rf_monte_carlo_test[
        "tournament_results"
    ][0]["runner_up"]
)

Number of simulations: 100
Stored tournaments: 100
First champion: Argentina
First runner-up: Spain


In [36]:
assert (
    rf_monte_carlo_test["n_simulations"]
    == 100
)

assert (
    len(
        rf_monte_carlo_test[
            "tournament_results"
        ]
    )
    == 100
)

assert (
    rf_monte_carlo_test["random_state"]
    == 42
)

assert (
    rf_monte_carlo_test["initial_teams"]
    == [
        team
        for fixture in test_tournament_fixtures
        for team in fixture
    ]
)

assert all(
    result["simulation_number"]
    == index
    for index, result in enumerate(
        rf_monte_carlo_test[
            "tournament_results"
        ],
        start=1
    )
)

assert all(
    result["champion"]
    in rf_monte_carlo_test["initial_teams"]
    for result in rf_monte_carlo_test[
        "tournament_results"
    ]
)

assert all(
    result["runner_up"]
    in rf_monte_carlo_test["initial_teams"]
    for result in rf_monte_carlo_test[
        "tournament_results"
    ]
)

assert all(
    result["champion"]
    != result["runner_up"]
    for result in rf_monte_carlo_test[
        "tournament_results"
    ]
)

assert all(
    len(result["rounds"]) == 3
    for result in rf_monte_carlo_test[
        "tournament_results"
    ]
)

print("Monte Carlo simulation test passed.")

Monte Carlo simulation test passed.


## Compute Tournament Probabilities

The complete Monte Carlo tournament results are transformed into team-level
probability estimates.

For each team, the function counts how often it advances to every knockout
stage, reaches the final, finishes as runner-up, and wins the tournament.

Each count is divided by the total number of simulations to estimate the
corresponding probability.

In [40]:
def compute_tournament_probabilities(
    monte_carlo_results
):
    """
    Compute team-level tournament probabilities from Monte Carlo results.

    Parameters
    ----------
    monte_carlo_results : dict
        Output generated by run_monte_carlo(), containing simulation metadata
        and the complete result of every simulated tournament.

    Returns
    -------
    pandas.DataFrame
        Team-level counts and probabilities for every tournament stage,
        including runner-up and championship results.
    """

    required_keys = {
        "n_simulations",
        "initial_teams",
        "tournament_results"
    }

    missing_keys = (
        required_keys
        - set(monte_carlo_results.keys())
    )

    if missing_keys:
        raise KeyError(
            "Missing Monte Carlo result keys: "
            f"{sorted(missing_keys)}"
        )

    n_simulations = monte_carlo_results[
        "n_simulations"
    ]

    initial_teams = [
        str(team)
        for team in monte_carlo_results[
            "initial_teams"
        ]
    ]

    tournament_results = monte_carlo_results[
        "tournament_results"
    ]

    if n_simulations <= 0:
        raise ValueError(
            "n_simulations must be greater than zero."
        )

    if len(tournament_results) != n_simulations:
        raise ValueError(
            "The number of stored tournament results does not "
            "match n_simulations."
        )

    stage_counts = {
        team: {}
        for team in initial_teams
    }

    runner_up_counts = {
        team: 0
        for team in initial_teams
    }

    champion_counts = {
        team: 0
        for team in initial_teams
    }

    for tournament_result in tournament_results:

        champion = str(
            tournament_result["champion"]
        )

        runner_up = str(
            tournament_result["runner_up"]
        )

        champion_counts[champion] += 1
        runner_up_counts[runner_up] += 1

        for round_result in tournament_result[
            "rounds"
        ]:

            qualified_teams = [
                str(team)
                for team in round_result[
                    "qualified_teams"
                ]
            ]

            number_of_qualified_teams = len(
                qualified_teams
            )

            if number_of_qualified_teams == 1:
                continue

            elif number_of_qualified_teams == 2:
                stage_name = "reach_final"

            elif number_of_qualified_teams == 4:
                stage_name = "reach_semifinal"

            elif number_of_qualified_teams == 8:
                stage_name = "reach_quarterfinal"

            else:
                stage_name = (
                    f"reach_round_of_"
                    f"{number_of_qualified_teams}"
                )

            for team in qualified_teams:

                current_count = stage_counts[
                    team
                ].get(stage_name, 0)

                stage_counts[team][
                    stage_name
                ] = current_count + 1

    all_stage_names = sorted(
        {
            stage_name
            for team_counts in stage_counts.values()
            for stage_name in team_counts
        }
    )

    records = []

    for team in initial_teams:

        record = {
            "team": team
        }

        for stage_name in all_stage_names:

            stage_count = stage_counts[
                team
            ].get(stage_name, 0)

            record[
                f"{stage_name}_count"
            ] = stage_count

            record[
                f"{stage_name}_probability"
            ] = (
                stage_count
                / n_simulations
            )

        record["runner_up_count"] = (
            runner_up_counts[team]
        )

        record["runner_up_probability"] = (
            runner_up_counts[team]
            / n_simulations
        )

        record["champion_count"] = (
            champion_counts[team]
        )

        record["champion_probability"] = (
            champion_counts[team]
            / n_simulations
        )

        records.append(record)

    probability_table = pd.DataFrame(
        records
    )

    probability_table = (
        probability_table
        .sort_values(
            by=[
                "champion_probability",
                "runner_up_probability"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )

    return probability_table

### Test Tournament Probability Computation


In [41]:
rf_probability_table_test = (
    compute_tournament_probabilities(
        rf_monte_carlo_test
    )
)

rf_probability_table_test

,team,reach_final_count,reach_final_probability,reach_semifinal_count,reach_semifinal_probability,runner_up_count,runner_up_probability,champion_count,champion_probability
0,Netherlands,30,0.30,50,0.50,12,0.12,18,0.18
1,Brazil,31,0.31,53,0.53,16,0.16,15,0.15
2,Spain,30,0.30,49,0.49,16,0.16,14,0.14
3,Argentina,20,0.20,47,0.47,7,0.07,13,0.13
4,Portugal,21,0.21,51,0.51,9,0.09,12,0.12
5,Germany,19,0.19,50,0.50,8,0.08,11,0.11
6,France,18,0.18,39,0.39,8,0.08,10,0.10
7,England,31,0.31,61,0.61,24,0.24,7,0.07


In [42]:
rf_probability_display_test = (
    rf_probability_table_test[
        [
            "team",
            "reach_semifinal_probability",
            "reach_final_probability",
            "runner_up_probability",
            "champion_probability"
        ]
    ]
    .copy()
)

probability_columns = [
    column
    for column in rf_probability_display_test.columns
    if column.endswith("_probability")
]

rf_probability_display_test[
    probability_columns
] = (
    rf_probability_display_test[
        probability_columns
    ]
    * 100
).round(2)

rf_probability_display_test

,team,reach_semifinal_probability,reach_final_probability,runner_up_probability,champion_probability
0,Netherlands,50.0,30.0,12.0,18.0
1,Brazil,53.0,31.0,16.0,15.0
2,Spain,49.0,30.0,16.0,14.0
3,Argentina,47.0,20.0,7.0,13.0
4,Portugal,51.0,21.0,9.0,12.0
5,Germany,50.0,19.0,8.0,11.0
6,France,39.0,18.0,8.0,10.0
7,England,61.0,31.0,24.0,7.0


In [43]:
assert isinstance(
    rf_probability_table_test,
    pd.DataFrame
)

assert len(
    rf_probability_table_test
) == len(
    rf_monte_carlo_test["initial_teams"]
)

assert (
    rf_probability_table_test["team"].nunique()
    == len(
        rf_monte_carlo_test["initial_teams"]
    )
)

assert (
    rf_probability_table_test[
        "champion_count"
    ].sum()
    == rf_monte_carlo_test[
        "n_simulations"
    ]
)

assert (
    rf_probability_table_test[
        "runner_up_count"
    ].sum()
    == rf_monte_carlo_test[
        "n_simulations"
    ]
)

assert np.isclose(
    rf_probability_table_test[
        "champion_probability"
    ].sum(),
    1.0
)

assert np.isclose(
    rf_probability_table_test[
        "runner_up_probability"
    ].sum(),
    1.0
)

assert np.isclose(
    rf_probability_table_test[
        "reach_final_probability"
    ].sum(),
    2.0
)

assert np.isclose(
    rf_probability_table_test[
        "reach_semifinal_probability"
    ].sum(),
    4.0
)

assert rf_probability_table_test[
    "champion_probability"
].between(
    0.0,
    1.0
).all()

assert rf_probability_table_test[
    "runner_up_probability"
].between(
    0.0,
    1.0
).all()

assert (
    rf_probability_table_test[
        "champion_count"
    ]
    <= rf_probability_table_test[
        "reach_final_count"
    ]
).all()

assert (
    rf_probability_table_test[
        "runner_up_count"
    ]
    <= rf_probability_table_test[
        "reach_final_count"
    ]
).all()

assert (
    rf_probability_table_test[
        "champion_count"
    ]
    + rf_probability_table_test[
        "runner_up_count"
    ]
    == rf_probability_table_test[
        "reach_final_count"
    ]
).all()

print(
    "Tournament probability computation test passed."
)

Tournament probability computation test passed.


# Random Forest Simulation

The Random Forest model is used to simulate the complete knockout tournament
multiple times through Monte Carlo simulation.

The resulting tournament outcomes are transformed into team-level probability
estimates for comparison with the remaining models.

### Run model

In [ ]:
rf_monte_carlo_results = run_monte_carlo(
    initial_fixtures=test_tournament_fixtures,
    model=rf_model,
    current_team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
    feature_names=rf_features,
    target_classes=rf_artifact["target_classes"],
    n_simulations=1000,
    random_state=42
)

### Compute Probabilities

In [45]:
rf_probability_table = (
    compute_tournament_probabilities(
        rf_monte_carlo_results
    )
)

### Display

In [46]:
rf_probability_display = (
    rf_probability_table[
        [
            "team",
            "reach_semifinal_probability",
            "reach_final_probability",
            "runner_up_probability",
            "champion_probability"
        ]
    ]
    .copy()
)

probability_columns = [
    column
    for column in rf_probability_display.columns
    if column.endswith("_probability")
]

rf_probability_display[
    probability_columns
] = (
    rf_probability_display[
        probability_columns
    ]
    * 100
).round(2)

rf_probability_display

,team,reach_semifinal_probability,reach_final_probability,runner_up_probability,champion_probability
0,England,54.6,29.0,14.2,14.8
1,Portugal,54.1,26.6,12.1,14.5
2,Netherlands,55.9,28.7,14.6,14.1
3,Brazil,53.9,25.9,13.4,12.5
4,France,45.4,25.6,13.3,12.3
5,Spain,45.9,23.2,11.8,11.4
6,Germany,44.1,21.5,10.2,11.3
7,Argentina,46.1,19.5,10.4,9.1


### Validation

In [47]:
assert len(
    rf_probability_table
) == len(test_tournament_fixtures) * 2

assert np.isclose(
    rf_probability_table[
        "champion_probability"
    ].sum(),
    1.0
)

assert np.isclose(
    rf_probability_table[
        "runner_up_probability"
    ].sum(),
    1.0
)

print(
    "Random Forest simulation completed successfully."
)

Random Forest simulation completed successfully.


# XGBoost Simulation

The XGBoost model is used to simulate the complete knockout tournament
multiple times through Monte Carlo simulation.

The resulting tournament outcomes are transformed into team-level probability
estimates for comparison with the remaining models.

### Run model

In [63]:
xgb_monte_carlo_results = run_monte_carlo(
    initial_fixtures=test_tournament_fixtures,
    model=xgb_model,
    current_team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
    feature_names=xgb_features,
    target_classes=[-1, 0, 1],
    n_simulations=1000,
    random_state=42
)

### Compute Probabilities

In [64]:
xgb_probability_table = (
    compute_tournament_probabilities(
        xgb_monte_carlo_results
    )
)

### Display

In [65]:
xgb_probability_display = (
    rf_probability_table[
        [
            "team",
            "reach_semifinal_probability",
            "reach_final_probability",
            "runner_up_probability",
            "champion_probability"
        ]
    ]
    .copy()
)

probability_columns = [
    column
    for column in xgb_probability_display.columns
    if column.endswith("_probability")
]

rf_probability_display[
    probability_columns
] = (
    xgb_probability_display[
        probability_columns
    ]
    * 100
).round(2)

xgb_probability_display

,team,reach_semifinal_probability,reach_final_probability,runner_up_probability,champion_probability
0,England,0.546,0.290,0.142,0.148
1,Portugal,0.541,0.266,0.121,0.145
2,Netherlands,0.559,0.287,0.146,0.141
3,Brazil,0.539,0.259,0.134,0.125
4,France,0.454,0.256,0.133,0.123
5,Spain,0.459,0.232,0.118,0.114
6,Germany,0.441,0.215,0.102,0.113
7,Argentina,0.461,0.195,0.104,0.091


### Validation

In [66]:
assert len(
    xgb_probability_table
) == len(test_tournament_fixtures) * 2

assert np.isclose(
    xgb_probability_table[
        "champion_probability"
    ].sum(),
    1.0
)

assert np.isclose(
    xgb_probability_table[
        "runner_up_probability"
    ].sum(),
    1.0
)

print(
    "XGBoost simulation completed successfully."
)

XGBoost simulation completed successfully.


# Ensemble Simulation

A soft-voting ensemble combines the match-level probability distributions
generated by the Random Forest and XGBoost models.

The probabilities are aligned to the canonical outcome classes and averaged
before each Monte Carlo sample is drawn.

This allows the complete tournament simulation to be driven by the combined
probabilistic assessment of both models rather than by averaging their final
tournament tables.

### Ensemble Model Wrapper

In [67]:
class SoftVotingEnsembleModel:
    """
    Soft-voting ensemble for models with potentially different feature
    schemas and internal class encodings.

    Each model generates a probability distribution. These probabilities are
    aligned to the canonical target classes and combined through a weighted
    average.
    """

    def __init__(
        self,
        models,
        feature_names,
        decoded_classes,
        weights=None,
        canonical_classes=(-1, 0, 1)
    ):
        """
        Parameters
        ----------
        models : list
            Trained classification models implementing predict_proba().

        feature_names : list of list-like
            Ordered feature schema expected by each model.

        decoded_classes : list of list-like
            Canonical classes corresponding, in order, to each model's
            predict_proba() columns.

        weights : list-like, optional
            Relative contribution of each model. Equal weights are used by
            default.

        canonical_classes : tuple, default=(-1, 0, 1)
            Shared outcome representation:
            -1 = Away Win
             0 = Draw
             1 = Home Win
        """

        if not models:
            raise ValueError(
                "At least one model is required."
            )

        if not (
            len(models)
            == len(feature_names)
            == len(decoded_classes)
        ):
            raise ValueError(
                "models, feature_names, and decoded_classes "
                "must have the same length."
            )

        self.models = list(models)

        self.feature_names = [
            list(model_features)
            for model_features in feature_names
        ]

        self.decoded_classes = [
            list(model_classes)
            for model_classes in decoded_classes
        ]

        self.classes_ = np.array(
            canonical_classes,
            dtype=int
        )

        if weights is None:
            weights = np.ones(
                len(self.models),
                dtype=float
            )

        self.weights = np.asarray(
            weights,
            dtype=float
        )

        if len(self.weights) != len(self.models):
            raise ValueError(
                "weights must contain one value per model."
            )

        if np.any(self.weights < 0):
            raise ValueError(
                "weights cannot contain negative values."
            )

        if np.isclose(self.weights.sum(), 0.0):
            raise ValueError(
                "At least one ensemble weight must be positive."
            )

        self.weights = (
            self.weights
            / self.weights.sum()
        )

    def predict_proba(self, X):
        """
        Generate weighted ensemble probabilities.
        """

        ensemble_probabilities = np.zeros(
            shape=(
                len(X),
                len(self.classes_)
            ),
            dtype=float
        )

        canonical_index = {
            int(target_class): index
            for index, target_class in enumerate(
                self.classes_
            )
        }

        for (
            model,
            model_features,
            model_decoded_classes,
            model_weight
        ) in zip(
            self.models,
            self.feature_names,
            self.decoded_classes,
            self.weights
        ):

            model_input = X.loc[
                :,
                model_features
            ].copy()

            model_probabilities = (
                model.predict_proba(
                    model_input
                )
            )

            if (
                model_probabilities.shape[1]
                != len(model_decoded_classes)
            ):
                raise ValueError(
                    "Model probability columns do not match "
                    "its decoded class configuration."
                )

            aligned_probabilities = np.zeros_like(
                ensemble_probabilities
            )

            for probability_index, target_class in enumerate(
                model_decoded_classes
            ):

                target_class = int(target_class)

                if target_class not in canonical_index:
                    raise ValueError(
                        "Unsupported decoded target class: "
                        f"{target_class}"
                    )

                aligned_probabilities[
                    :,
                    canonical_index[target_class]
                ] = model_probabilities[
                    :,
                    probability_index
                ]

            ensemble_probabilities += (
                model_weight
                * aligned_probabilities
            )

        probability_sums = (
            ensemble_probabilities.sum(
                axis=1,
                keepdims=True
            )
        )

        if np.any(
            np.isclose(
                probability_sums,
                0.0
            )
        ):
            raise ValueError(
                "The ensemble generated an invalid "
                "zero-sum probability distribution."
            )

        return (
            ensemble_probabilities
            / probability_sums
        )

### Configure Ensemble

In [68]:
ensemble_features = list(
    dict.fromkeys(
        list(rf_features)
        + list(xgb_features)
    )
)

print(
    "Random Forest features:",
    len(rf_features)
)

print(
    "XGBoost features:",
    len(xgb_features)
)

print(
    "Ensemble features:",
    len(ensemble_features)
)

Random Forest features: 124
XGBoost features: 124
Ensemble features: 125


In [69]:
ensemble_model = SoftVotingEnsembleModel(
    models=[
        rf_model,
        xgb_model
    ],
    feature_names=[
        rf_features,
        xgb_features
    ],
    decoded_classes=[
        [-1, 0, 1],
        [-1, 0, 1]
    ],
    weights=[
        0.5,
        0.5
    ]
)

### Test Ensemble Probabilities


In [70]:
ensemble_match_test = simulate_match(
    home_team="Argentina",
    away_team="Brazil",
    model=ensemble_model,
    current_team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
    feature_names=ensemble_features,
    target_classes=[-1, 0, 1],
    rng=np.random.default_rng(42)
)

print(
    "Simulated result:",
    ensemble_match_test["simulated_result"]
)

print(
    "Winner:",
    ensemble_match_test["winner"]
)

print(
    "Probabilities:",
    ensemble_match_test["probabilities"]
)

Simulated result: Home Win
Winner: Argentina
Probabilities: {'Home Win': 0.2840382900533207, 'Draw': 0.2893327730198743, 'Away Win': 0.4266289369268049}


### Validation

In [71]:
assert np.isclose(
    sum(
        ensemble_match_test[
            "probabilities"
        ].values()
    ),
    1.0
)

assert ensemble_match_test[
    "simulated_result"
] in {
    "Home Win",
    "Draw",
    "Away Win"
}

assert ensemble_match_test[
    "winner"
] in {
    "Argentina",
    "Brazil",
    None
}

print(
    "Ensemble match probability test passed."
)

Ensemble match probability test passed.


### Ensemble Monte Carlo

In [72]:
ensemble_monte_carlo_results = run_monte_carlo(
    initial_fixtures=test_tournament_fixtures,
    model=ensemble_model,
    current_team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
    feature_names=ensemble_features,
    target_classes=[-1, 0, 1],
    n_simulations=1000,
    random_state=42
)

### Compute Ensemble Probabilities

In [73]:
ensemble_probability_table = (
    compute_tournament_probabilities(
        ensemble_monte_carlo_results
    )
)

### Display

In [74]:
ensemble_probability_display = (
    ensemble_probability_table[
        [
            "team",
            "reach_semifinal_probability",
            "reach_final_probability",
            "runner_up_probability",
            "champion_probability"
        ]
    ]
    .copy()
)

ensemble_probability_columns = [
    column
    for column in ensemble_probability_display.columns
    if column.endswith("_probability")
]

ensemble_probability_display[
    ensemble_probability_columns
] = (
    ensemble_probability_display[
        ensemble_probability_columns
    ]
    * 100
).round(2)

ensemble_probability_display

,team,reach_semifinal_probability,reach_final_probability,runner_up_probability,champion_probability
0,Netherlands,60.5,36.8,17.1,19.7
1,Portugal,65.2,26.1,10.1,16.0
2,Germany,39.5,24.0,9.8,14.2
3,France,46.9,30.1,16.1,14.0
4,Brazil,58.6,26.6,14.6,12.0
5,England,53.1,27.6,17.9,9.7
6,Spain,34.8,13.1,5.6,7.5
7,Argentina,41.4,15.7,8.8,6.9


### Validation

In [75]:
assert (
    len(ensemble_probability_table)
    == len(test_tournament_fixtures) * 2
)

assert np.isclose(
    ensemble_probability_table[
        "champion_probability"
    ].sum(),
    1.0
)

assert np.isclose(
    ensemble_probability_table[
        "runner_up_probability"
    ].sum(),
    1.0
)

assert np.isclose(
    ensemble_probability_table[
        "reach_final_probability"
    ].sum(),
    2.0
)

assert np.isclose(
    ensemble_probability_table[
        "reach_semifinal_probability"
    ].sum(),
    4.0
)

print(
    "Ensemble simulation completed successfully."
)

Ensemble simulation completed successfully.


# Model Comparison

The Random Forest, XGBoost, and soft-voting ensemble probability tables are
combined into a single comparison report.

The report highlights how each modeling approach estimates tournament
progression and championship probability for every team.

In [76]:
rf_comparison = (
    rf_probability_table[
        [
            "team",
            "reach_semifinal_probability",
            "reach_final_probability",
            "champion_probability"
        ]
    ]
    .rename(
        columns={
            "reach_semifinal_probability":
                "rf_reach_semifinal_probability",
            "reach_final_probability":
                "rf_reach_final_probability",
            "champion_probability":
                "rf_champion_probability"
        }
    )
)

xgb_comparison = (
    xgb_probability_table[
        [
            "team",
            "reach_semifinal_probability",
            "reach_final_probability",
            "champion_probability"
        ]
    ]
    .rename(
        columns={
            "reach_semifinal_probability":
                "xgb_reach_semifinal_probability",
            "reach_final_probability":
                "xgb_reach_final_probability",
            "champion_probability":
                "xgb_champion_probability"
        }
    )
)

ensemble_comparison = (
    ensemble_probability_table[
        [
            "team",
            "reach_semifinal_probability",
            "reach_final_probability",
            "champion_probability"
        ]
    ]
    .rename(
        columns={
            "reach_semifinal_probability":
                "ensemble_reach_semifinal_probability",
            "reach_final_probability":
                "ensemble_reach_final_probability",
            "champion_probability":
                "ensemble_champion_probability"
        }
    )
)

model_comparison = (
    rf_comparison
    .merge(
        xgb_comparison,
        on="team",
        how="inner"
    )
    .merge(
        ensemble_comparison,
        on="team",
        how="inner"
    )
)

## Add Model Disagreement

In [77]:
model_comparison[
    "champion_probability_range"
] = (
    model_comparison[
        [
            "rf_champion_probability",
            "xgb_champion_probability",
            "ensemble_champion_probability"
        ]
    ]
    .max(axis=1)
    -
    model_comparison[
        [
            "rf_champion_probability",
            "xgb_champion_probability",
            "ensemble_champion_probability"
        ]
    ]
    .min(axis=1)
)

## Sort Report

In [78]:
model_comparison = (
    model_comparison
    .sort_values(
        by="ensemble_champion_probability",
        ascending=False
    )
    .reset_index(drop=True)
)

## Display Comparison

In [79]:
model_comparison_display = (
    model_comparison[
        [
            "team",
            "rf_champion_probability",
            "xgb_champion_probability",
            "ensemble_champion_probability",
            "champion_probability_range"
        ]
    ]
    .copy()
)

comparison_probability_columns = [
    column
    for column in model_comparison_display.columns
    if column.endswith("_probability")
    or column.endswith("_range")
]

model_comparison_display[
    comparison_probability_columns
] = (
    model_comparison_display[
        comparison_probability_columns
    ]
    * 100
).round(2)

model_comparison_display

,team,rf_champion_probability,xgb_champion_probability,ensemble_champion_probability,champion_probability_range
0,Netherlands,14.1,25.4,19.7,11.3
1,Portugal,14.5,15.4,16.0,1.5
2,Germany,11.3,15.3,14.2,4.0
3,France,12.3,11.7,14.0,2.3
4,Brazil,12.5,14.0,12.0,2.0
5,England,14.8,8.1,9.7,6.7
6,Spain,11.4,4.2,7.5,7.2
7,Argentina,9.1,5.9,6.9,3.2


## Validation

In [80]:
assert (
    len(model_comparison)
    == len(test_tournament_fixtures) * 2
)

assert (
    model_comparison["team"].nunique()
    == len(test_tournament_fixtures) * 2
)

assert np.isclose(
    model_comparison[
        "rf_champion_probability"
    ].sum(),
    1.0
)

assert np.isclose(
    model_comparison[
        "xgb_champion_probability"
    ].sum(),
    1.0
)

assert np.isclose(
    model_comparison[
        "ensemble_champion_probability"
    ].sum(),
    1.0
)

assert model_comparison[
    "champion_probability_range"
].between(
    0.0,
    1.0
).all()

print(
    "Model comparison report completed successfully."
)

Model comparison report completed successfully.


# Champion Probability Ranking

The ensemble model provides the final tournament probability estimates by
combining the Random Forest and XGBoost predictions through soft voting.

Teams are ranked according to their estimated probability of winning the
tournament.

In [81]:
champion_ranking = (
    model_comparison[
        [
            "team",
            "rf_champion_probability",
            "xgb_champion_probability",
            "ensemble_champion_probability"
        ]
    ]
    .copy()
)

champion_ranking = (
    champion_ranking
    .sort_values(
        by="ensemble_champion_probability",
        ascending=False
    )
    .reset_index(drop=True)
)

champion_ranking.insert(
    0,
    "rank",
    np.arange(
        1,
        len(champion_ranking) + 1
    )
)

### Display

In [82]:
champion_ranking_display = (
    champion_ranking.copy()
)

probability_columns = [
    column
    for column in champion_ranking_display.columns
    if column.endswith("_probability")
]

champion_ranking_display[
    probability_columns
] = (
    champion_ranking_display[
        probability_columns
    ]
    * 100
).round(2)

champion_ranking_display

,rank,team,rf_champion_probability,xgb_champion_probability,ensemble_champion_probability
0,1,Netherlands,14.1,25.4,19.7
1,2,Portugal,14.5,15.4,16.0
2,3,Germany,11.3,15.3,14.2
3,4,France,12.3,11.7,14.0
4,5,Brazil,12.5,14.0,12.0
5,6,England,14.8,8.1,9.7
6,7,Spain,11.4,4.2,7.5
7,8,Argentina,9.1,5.9,6.9


### Summary

In [83]:
top_team = champion_ranking.iloc[0]

print(
    "Most likely champion:",
    top_team["team"]
)

print(
    "Ensemble probability:",
    f"{top_team['ensemble_champion_probability']:.2%}"
)

Most likely champion: Netherlands
Ensemble probability: 19.70%


### Validation

In [84]:
assert (
    champion_ranking["rank"]
    == np.arange(
        1,
        len(champion_ranking) + 1
    )
).all()

assert champion_ranking[
    "ensemble_champion_probability"
].is_monotonic_decreasing

assert np.isclose(
    champion_ranking[
        "ensemble_champion_probability"
    ].sum(),
    1.0
)

print(
    "Champion probability ranking completed successfully."
)

Champion probability ranking completed successfully.


# Discussion

### English

This notebook completes the probabilistic tournament simulation component of
the World Cup Predictor project.

A modular Monte Carlo simulation engine was developed to estimate tournament
progression and championship probabilities from match-level prediction models.
The simulation pipeline remains independent of the underlying predictive
algorithm, allowing any compatible model implementing a probability interface
to be incorporated without modifying the simulation logic.

Three predictive approaches were evaluated:

- Random Forest,
- XGBoost,
- and a soft-voting ensemble combining both models.

The ensemble architecture demonstrates the flexibility of the modular design,
showing that multiple predictive models can be integrated while preserving a
single simulation workflow.

Although the current system represents football primarily through aggregated
statistical event features, the project now provides a complete end-to-end
pipeline capable of:

- constructing prediction-ready feature vectors,
- loading trained models,
- simulating complete knockout tournaments,
- estimating advancement probabilities,
- and comparing multiple predictive approaches under identical tournament
  conditions.

The Future Work section outlines several research directions intended to move
the project beyond purely statistical representations toward richer tactical,
spatial, temporal, and player-role aware models.

Overall, this notebook concludes the first complete implementation of the
World Cup Predictor simulation framework and establishes a solid modular
foundation for future development.

### Español


Esta notebook completa el componente de simulación probabilística de torneos
del proyecto World Cup Predictor.

Se desarrolló un motor de simulación Monte Carlo modular capaz de estimar
probabilidades de avance y de obtención del campeonato a partir de modelos de
predicción a nivel de partido. El motor permanece completamente desacoplado
del algoritmo predictivo utilizado, permitiendo incorporar cualquier modelo
compatible que implemente una interfaz probabilística sin modificar la lógica
de simulación.

Se evaluaron tres enfoques predictivos:

- Random Forest,
- XGBoost,
- y un ensemble de tipo Soft Voting que combina ambos modelos.

La arquitectura del ensemble demuestra la flexibilidad del diseño modular,
permitiendo integrar múltiples modelos predictivos dentro del mismo flujo de
simulación.

Si bien el sistema actual representa el fútbol principalmente mediante
estadísticas agregadas de eventos, el proyecto dispone ahora de un pipeline
completo capaz de:

- construir vectores listos para predicción,
- cargar modelos previamente entrenados,
- simular torneos completos de eliminación directa,
- estimar probabilidades de avance,
- y comparar diferentes enfoques predictivos bajo las mismas condiciones de
  simulación.

La sección de Trabajo Futuro presenta diversas líneas de investigación que
permitirán evolucionar el proyecto hacia modelos capaces de comprender mejor
los aspectos tácticos, espaciales, temporales y los distintos roles de los
jugadores.

En conjunto, esta notebook marca el cierre de la primera implementación
completa del framework de simulación del World Cup Predictor y establece una
base modular sólida para las siguientes fases del proyecto.

# Future Work

The current project establishes a complete Phase I pipeline for feature
construction, model inference, tournament simulation, and probabilistic model
comparison.

However, the present system still represents football primarily through
aggregated statistical features. Future development should progressively move
the project toward a richer representation of players, team structure,
match evolution, and tactical behavior.


## Player Database Expansion

The player databases should be expanded using regional competitions,
continental tournaments, international qualifiers, domestic leagues, and
other available match sources.

This is especially important for national teams with low current roster
coverage, where many players do not yet have sufficient historical event
data.

The objective is to build complete and continuously updated databases that
provide reliable statistical profiles for every relevant player and team.

## Player Role Efficiency



Future player evaluation should not be based only on total event production.

Each player should be evaluated according to the responsibilities of their
own position and tactical role.

For example, a defender, defensive midfielder, winger, and striker should not
be assessed using the same criteria. Their efficiency should be measured
relative to the actions expected from their role, such as:

- defensive contribution,
- ball progression,
- passing reliability,
- chance creation,
- pressing activity,
- finishing efficiency,
- spatial occupation,
- and involvement in different phases of play.

This would allow the project to distinguish between high-volume players and
players who perform their assigned role efficiently.



## Collective Role Balance

Team strength should be modeled as more than the sum of individual player
statistics.

The system should evaluate how individual roles combine to produce a balanced
and functional team structure.

This would prevent unrealistic conclusions such as assuming that selecting
eleven attacking players necessarily increases the probability of scoring or
winning.

A future team representation should account for:

- positional balance,
- defensive coverage,
- midfield control,
- attacking width and depth,
- role complementarity,
- ball progression structure,
- pressing organization,
- and tactical compatibility between players.

The objective is to estimate the collective capacity of the lineup while
respecting the tactical constraints of football.



## Broader Historical Learning


Future models should learn from the largest reliable collection of matches and
competitions available.

This should include:

- FIFA World Cups,
- continental championships,
- international qualifiers,
- regional competitions,
- friendly matches when appropriate,
- domestic leagues,
- and club-level international competitions.

The expansion should not consist only of adding more rows. Competition quality,
historical relevance, temporal distance, tactical context, and data reliability
should also be considered when determining the importance of each match.



## Temporal Match Representation



The current feature vectors primarily summarize the final accumulated state of
a match.

Future versions should model how a match evolves over time.

Matches may be divided into temporal segments or dynamic windows in order to
capture:

- changes in possession,
- pressure phases,
- attacking momentum,
- territorial dominance,
- shot creation,
- defensive reactions,
- substitutions,
- score-state effects,
- tactical adjustments,
- and late-match behavior.

This would allow the model to learn not only what happened during a match, but
also how the match developed.

StatsBomb 360 data would be especially valuable for this stage because it can
provide richer spatial and contextual information around each event.



## Tactical Feature Engineering



New features should move the models beyond isolated statistical totals and
toward the recognition of tactical patterns.

Potential future features include:

- recent team momentum,
- player and team form,
- pass-recipient distributions,
- passing networks,
- progression routes,
- repeated passing sequences,
- build-up patterns,
- direct versus positional attacks,
- width and central occupation,
- pressing intensity,
- defensive block behavior,
- counterattacking frequency,
- transition efficiency,
- possession-value evolution,
- and interactions between specific player roles.

Pass-recipient information may be particularly useful for identifying the
types of attacking sequences a team constructs, rather than considering every
completed pass as an equivalent event.



## Spatial and 360-Degree Analysis



A future tactical layer should incorporate player locations, surrounding
pressure, passing options, occupied zones, and defensive structures.

This would make it possible to analyze:

- space creation,
- line breaking,
- numerical superiority,
- compactness,
- off-ball movement,
- defensive shape,
- passing lane availability,
- and the relationship between the ball carrier and nearby teammates or
  opponents.

Such information would significantly improve the model's ability to recognize
football situations instead of relying only on aggregated numerical outcomes.



## Probability Calibration and Model Validation



Tournament probabilities should eventually be calibrated and validated using a
larger out-of-sample historical framework.

Future work should compare:

- predicted match probabilities,
- observed frequencies,
- tournament advancement probabilities,
- model confidence,
- and ensemble stability.

This would help determine whether a model that achieves higher classification
accuracy also produces more reliable probabilities.



## Group Stage and Full Tournament Simulation



The current simulation engine focuses on knockout tournaments.

A future version should include:

- group-stage match simulation,
- points and standings,
- goal-based tie-break rules,
- qualification scenarios,
- bracket construction,
- and complete World Cup tournament simulation.

The group-stage engine should remain separated from the knockout engine so that
both components preserve clear responsibilities and can be validated
independently.



## Continuous Updating



During the 2026 World Cup, the project should support continuous data updates.

New matches should be incorporated into the databases, player vectors, team
vectors, and prediction pipeline as the tournament progresses.

The long-term objective is to create a system that can update its understanding
of:

- player form,
- team performance,
- tactical tendencies,
- roster changes,
- injuries,
- and tournament momentum.


## Long-Term Objective



The long-term objective is to transform the project from a statistical
match-result predictor into a football intelligence system.

Rather than learning only that certain numerical differences correlate with
victory, the models should progressively learn:

- why a team creates advantages,
- how players perform within their roles,
- how roles interact collectively,
- how tactical structures evolve,
- and how match dynamics influence the final result.

Phase I provides the modular foundation required to pursue that objective.

# Trabajo Futuro

El proyecto actual establece un flujo de trabajo completo de Fase I para la construcción de características, la inferencia de modelos, la simulación de torneos y la comparación probabilística de modelos.

Sin embargo, el sistema actual sigue representando el fútbol principalmente mediante características estadísticas agregadas. El desarrollo futuro debería orientar progresivamente el proyecto hacia una representación más rica de los jugadores, la estructura del equipo, la evolución del partido y el comportamiento táctico.

## Expansión de la Base de Datos de Jugadores

Las bases de datos de jugadores deberían ampliarse utilizando competiciones
regionales, torneos continentales, eliminatorias internacionales, ligas
domésticas y cualquier otra fuente de partidos disponible.

Esto resulta especialmente importante para las selecciones con baja cobertura
del plantel actual, donde muchos jugadores todavía no poseen suficiente
historial de eventos.

El objetivo es construir bases de datos completas y continuamente
actualizadas que permitan generar perfiles estadísticos confiables para cada
jugador y cada selección.

## Eficiencia Individual según el Rol del Jugador

La evaluación de un jugador no debería basarse únicamente en la cantidad total
de eventos que produce.

Cada futbolista debería ser analizado según las responsabilidades propias de
su posición y de su rol táctico.

Por ejemplo, un defensor central, un mediocampista defensivo, un extremo y un
delantero no deberían evaluarse utilizando exactamente los mismos criterios.
Su eficiencia debería medirse respecto de las acciones esperadas para su rol,
como:

- contribución defensiva,
- progresión del balón,
- calidad del pase,
- generación de ocasiones,
- presión,
- eficacia de definición,
- ocupación de espacios,
- y participación en las distintas fases del juego.

Esto permitiría diferenciar jugadores con mucho volumen estadístico de
aquellos que realmente ejecutan su función de manera eficiente.

## Balance Colectivo de Roles

La fortaleza de un equipo debería modelarse como algo más que la simple suma
de las estadísticas individuales.

El sistema debería evaluar cómo los distintos roles interactúan para formar
una estructura táctica equilibrada y funcional.

De esta forma se evitarían conclusiones irreales, como asumir que incorporar
once delanteros necesariamente incrementa la probabilidad de convertir goles o
ganar partidos.

Una representación futura del equipo debería contemplar:

- equilibrio posicional,
- cobertura defensiva,
- control del mediocampo,
- amplitud y profundidad ofensiva,
- complementariedad entre roles,
- estructura de progresión del balón,
- organización de la presión,
- y compatibilidad táctica entre jugadores.

El objetivo es estimar la capacidad colectiva de una formación respetando las
restricciones tácticas propias del fútbol.

## Aprendizaje a partir de una Mayor Cantidad de Competiciones

Los futuros modelos deberían aprender utilizando la mayor colección confiable
de partidos y competiciones posible.

Esto incluye:

- Copas del Mundo,
- torneos continentales,
- eliminatorias internacionales,
- competiciones regionales,
- amistosos cuando resulten apropiados,
- ligas nacionales,
- y competiciones internacionales de clubes.

La ampliación no debería consistir únicamente en agregar más partidos. También
deberían considerarse la calidad de la competición, su relevancia histórica,
la distancia temporal, el contexto táctico y la confiabilidad de los datos
para determinar la importancia relativa de cada encuentro.

## Representación Temporal del Partido

Actualmente los vectores representan principalmente el estado acumulado final
de cada partido.

Las futuras versiones deberían modelar cómo evoluciona un encuentro a lo largo
del tiempo.

Los partidos podrían dividirse en segmentos temporales o ventanas dinámicas
para capturar:

- cambios en la posesión,
- fases de presión,
- momentos de dominio territorial,
- generación de ocasiones,
- respuestas defensivas,
- sustituciones,
- efecto del marcador,
- ajustes tácticos,
- y comportamiento durante los minutos finales.

Esto permitiría que el modelo aprenda no solamente qué ocurrió en un partido,
sino también cómo se desarrolló.

Los datos de StatsBomb 360 resultarán especialmente valiosos para esta etapa,
ya que incorporan contexto espacial alrededor de cada evento.

## Ingeniería de Variables Tácticas

Las futuras variables deberían alejar progresivamente a los modelos de simples
estadísticas agregadas para acercarlos al reconocimiento de patrones tácticos.

Algunas posibles variables incluyen:

- momentum reciente de cada equipo,
- estado de forma de jugadores y selecciones,
- distribución de destinatarios de los pases,
- redes de pases,
- rutas de progresión,
- secuencias repetidas de circulación,
- patrones de construcción,
- ataques directos frente a ataques posicionales,
- amplitud y ocupación del carril central,
- intensidad de la presión,
- comportamiento del bloque defensivo,
- frecuencia de contraataques,
- eficiencia en las transiciones,
- evolución del valor de la posesión,
- e interacciones entre distintos roles.

En particular, analizar a quién se dirige cada pase podría permitir reconocer
tipos de jugadas y patrones ofensivos, en lugar de considerar todos los pases
completados como eventos equivalentes.

## Análisis Espacial y Datos 360

Una futura capa táctica debería incorporar la ubicación de los jugadores, la
presión circundante, las opciones de pase, los espacios ocupados y las
estructuras defensivas.

Esto permitiría analizar aspectos como:

- generación de espacios,
- ruptura de líneas,
- superioridades numéricas,
- compactación defensiva,
- movimientos sin balón,
- forma del bloque defensivo,
- disponibilidad de líneas de pase,
- y la relación entre el poseedor del balón y los jugadores cercanos.

Este tipo de información incrementaría considerablemente la capacidad del
modelo para reconocer situaciones reales de juego, en lugar de depender
únicamente de estadísticas agregadas.

## Calibración de Probabilidades y Validación de Modelos

Las probabilidades obtenidas mediante las simulaciones deberían calibrarse y
validarse utilizando un conjunto histórico más amplio.

Los futuros trabajos deberían comparar:

- probabilidades predichas,
- frecuencias observadas,
- probabilidades de avance en el torneo,
- nivel de confianza del modelo,
- y estabilidad del ensemble.

Esto permitirá evaluar si un modelo con mayor precisión también genera
probabilidades mejor calibradas y más confiables.

## Simulación Completa del Mundial

Actualmente el motor de simulación está orientado exclusivamente a torneos de
eliminación directa.

Las futuras versiones deberían incorporar:

- simulación de la fase de grupos,
- sistema de puntos,
- criterios oficiales de desempate,
- escenarios de clasificación,
- construcción automática del cuadro eliminatorio,
- y simulación completa de una Copa del Mundo.

La fase de grupos debería mantenerse desacoplada del motor de eliminación
directa para conservar responsabilidades claras y facilitar su validación.

## Actualización Continua

Durante la Copa del Mundo 2026 el proyecto debería ser capaz de incorporar
automáticamente los nuevos partidos disputados.

Las bases de datos de jugadores, vectores individuales, vectores de equipo y
modelos de predicción deberían actualizarse a medida que avanza el torneo.

El objetivo a largo plazo es construir un sistema capaz de adaptar
continuamente su conocimiento sobre:

- estado de forma,
- rendimiento colectivo,
- tendencias tácticas,
- cambios de convocatoria,
- lesiones,
- y momentum competitivo.

## Objetivo a Largo Plazo

El objetivo final del proyecto es evolucionar desde un predictor estadístico
de resultados hacia un sistema de inteligencia futbolística.

Más allá de aprender que determinadas diferencias numéricas se correlacionan
con la victoria, el modelo debería comprender progresivamente:

- por qué un equipo genera ventajas,
- cómo rinden los jugadores dentro de su rol,
- cómo interactúan esos roles entre sí,
- cómo evolucionan las estructuras tácticas,
- y cómo la dinámica del partido influye sobre el resultado final.

La Fase I establece la arquitectura modular necesaria para comenzar ese
camino y servir como base para futuras investigaciones y desarrollos.

# Conclusion

### English

This notebook concludes Phase I of the World Cup Predictor project.

Throughout the first phase, the project evolved from raw event data into a
complete prediction framework capable of generating player vectors, team
representations, match feature vectors, machine learning models, automated
prediction pipelines, and probabilistic tournament simulations.

The resulting architecture emphasizes modularity, reproducibility, and
extensibility. Each stage of the pipeline was designed as an independent
component, allowing future improvements to individual modules without
requiring extensive modifications to the rest of the system.

Although substantial opportunities remain for expanding the databases,
improving feature engineering, incorporating tactical information, and
developing richer football representations, the current implementation already
constitutes a functional and fully reproducible prediction system.

Future phases will focus on expanding both the available data and the
representational capacity of the models, progressively transforming the
project from a statistical predictor into a football intelligence framework.

Phase I successfully establishes the technical foundation required to pursue
that long-term objective.

### Español

Esta notebook marca el cierre de la Fase I del proyecto World Cup Predictor.

A lo largo de esta primera etapa, el proyecto evolucionó desde datos brutos de
eventos hasta convertirse en un sistema completo capaz de generar vectores de
jugadores, representaciones de equipos, vectores de partidos, modelos de
Machine Learning, pipelines automáticos de predicción y simulaciones
probabilísticas de torneos.

La arquitectura desarrollada prioriza la modularidad, la reproducibilidad y
la facilidad de expansión. Cada etapa del pipeline fue diseñada como un
componente independiente, permitiendo mejorar módulos individuales sin
necesidad de modificar el resto del sistema.

Si bien todavía existen numerosas oportunidades para ampliar las bases de
datos, enriquecer las variables, incorporar información táctica y desarrollar
representaciones más complejas del fútbol, la implementación actual ya
constituye un sistema funcional y completamente reproducible de predicción.

Las siguientes fases estarán orientadas a expandir tanto la cantidad y calidad
de los datos disponibles como la capacidad de representación de los modelos,
con el objetivo de evolucionar progresivamente desde un predictor estadístico
hacia un sistema de inteligencia futbolística.

La Fase I establece con éxito la base técnica necesaria para avanzar hacia ese
objetivo de largo plazo.